# 🎵 SongMind AI: Multimodal Emotion-Based Song Recommender

**Author:** Pratham Yadav  
**Domain:** Computer Vision, Natural Language Processing, Machine Learning  

---

## 📌 Project Overview
**SongMind AI** is an end-to-end multimodal recommendation system that predicts user mood and serves real-time song recommendations from a combined Spotify dataset. The pipeline integrates:
- 📸 **Computer Vision (DeepFace):** Analyzes real-time facial expressions from live camera feed.
- 💬 **Natural Language Processing (DistilBERT Zero-Shot):** Interprets contextual emotion from user chat text.
- 🎯 **Multimodal Feature Fusion:** Merges facial valence/energy scores with text sentiment vectors.
- 📐 **Cosine Similarity Matching Engine:** Mathematically ranks top-matching songs using multi-dimensional feature vectors.

---

## 📂 Step 1: Environment Setup & Drive Storage Integration
Mount Google Drive to access persistent dataset files and test images.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📊 Step 2: Dataset Verification & Active Workspace Preparation
Load the pre-built `spotify_data_combined.csv` dataset from Google Drive and copy it to the active working directory for fast Streamlit inference.

In [ ]:
import pandas as pd
import os

# Path to persistent dataset in Drive
drive_csv_path = '/content/drive/MyDrive/SongmindAI/Data/spotify_data_combined.csv'

# Verify and load combined dataset
if os.path.exists(drive_csv_path):
    df_combined = pd.read_csv(drive_csv_path)
    print("✓ Loaded dataset directly from Google Drive!")
elif os.path.exists('spotify_data_combined.csv'):
    df_combined = pd.read_csv('spotify_data_combined.csv')
    print("✓ Loaded local working directory dataset!")
else:
    raise FileNotFoundError("Combined dataset not found. Ensure spotify_data_combined.csv exists in Google Drive.")

# Save to active workspace for Streamlit app access
df_combined.to_csv('spotify_data_combined.csv', index=False)
print(f"✓ Dataset active in workspace. Total Tracks: {df_combined.shape[0]}, Features: {df_combined.shape[1]}")
display(df_combined[['Song Name', 'Artists', 'valence', 'energy', 'mood']].head(5))

✓ Loaded dataset directly from Google Drive!
✓ Dataset active in workspace. Total Tracks: 276, Features: 13


,Song Name,Artists,valence,energy,mood
0,Best Hindi Motivational Poetry 2022,Kumar Rishi,0.44,0.37,Reflective / Spoken Word
1,Relation,Nikk,0.58,0.50,Happy / Upbeat
2,Temporary pyar,Kaka,0.63,0.50,Happy / Upbeat
3,Distance Love,Zehr Vibe,0.57,0.45,Romantic / Warm
4,Gal Karke,Inder Chahal,0.43,0.47,Intense / Angsty


## 👁️ Step 3: Computer Vision & Facial Expression Recognition (FER)
Install `DeepFace` and OpenCV libraries to analyze facial expressions and map visual emotions to Spotify audio feature spaces (`valence` and `energy`).

In [ ]:
!pip install deepface opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.8 MB/s eta 0:00:00


### 🧪 Testing Computer Vision Pipeline Standalone
Evaluate model accuracy on a sample test image.

In [ ]:
from deepface import DeepFace
import cv2

def analyze_facial_emotion(image_path):
    try:
        result = DeepFace.analyze(img_path=image_path, actions=['emotion'], enforce_detection=False)
        dominant_emotion = result[0]['dominant_emotion']
        emotion_scores = result[0]['emotion']

        print(f"Dominant Emotion Detected: {dominant_emotion.upper()}")
        print("Confidence Scores:", emotion_scores)
        return dominant_emotion
    except Exception as e:
        print("Error processing face:", e)
        return None

def map_to_dataset_mood(fer_emotion):
    mapping = {
        'happy': 'Happy / Upbeat',
        'sad': 'Melancholic / Heartbroken',
        'angry': 'Intense / Angsty',
        'disgust': 'Intense / Angsty',
        'neutral': 'Mellow / Reflective',
        'surprise': 'Energetic / Party',
        'fear': 'Intense / Angsty'
    }
    return mapping.get(fer_emotion, 'Mellow / Reflective')

# Test on sample image
image_file = '/content/drive/MyDrive/SongmindAI/Data/Test_Image/test_face_3.jpg'
detected_emotion = analyze_facial_emotion(image_file)
matched_mood = map_to_dataset_mood(detected_emotion)
print(f"\n=> Mapped Spotify Category: {matched_mood}")

26-07-25 20:40:15 - Directory /root/.deepface has been created
26-07-25 20:40:15 - Directory /root/.deepface/weights has been created
26-07-25 20:41:12 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 119MB/s]


Dominant Emotion Detected: ANGRY
Confidence Scores: {'angry': np.float32(92.84862), 'disgust': np.float32(1.5002327e-06), 'fear': np.float32(0.024880296), 'happy': np.float32(0.028941361), 'sad': np.float32(5.6009674), 'surprise': np.float32(4.803612e-06), 'neutral': np.float32(1.4965833)}

=> Mapped Spotify Category: Intense / Angsty


## 🛠️ Step 4: Installing Web Framework & NLP Transformer Libraries
Install `Streamlit`, Hugging Face `transformers`, `torch`, and `scikit-learn` to build the web frontend and text analysis backend.

In [ ]:
!pip install streamlit transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.0 MB/s eta 0:00:00


## 💻 Step 5: Constructing the Streamlit Web Application (`app.py`)
Write the unified multimodal recommender code into `app.py`. Uses `typeform/distilbert-base-uncased-mnli` for memory-efficient Zero-Shot NLP text classification and `scikit-learn` Cosine Similarity for vector matching.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import tempfile
import os
from deepface import DeepFace
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity

# ── 1. PAGE CONFIGURATION & SLEEK CSS STYLING ─────────────────────────────────
st.set_page_config(
    page_title="SongMind AI",
    page_icon="🎵",
    layout="centered",
    initial_sidebar_state="collapsed",
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@600;700;800&family=DM+Sans:wght@400;500;700&display=swap');

html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
}

.stApp {
    background: #0b0b12;
    color: #e2e8f0;
}

/* Hero Header */
.hero-title {
    font-family: 'Syne', sans-serif;
    font-size: 2.5rem;
    font-weight: 800;
    background: linear-gradient(135deg, #a855f7 0%, #6366f1 50%, #3b82f6 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 0.2rem;
}
.hero-sub {
    color: #94a3b8;
    font-size: 0.95rem;
    margin-bottom: 1.5rem;
}

/* Glassmorphism Card Containers */
.glass-card {
    background: rgba(22, 21, 35, 0.7);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 16px;
    padding: 18px;
    margin-bottom: 14px;
    backdrop-filter: blur(10px);
}

/* Status Badges */
.status-box {
    background: #141322;
    border-left: 4px solid #a855f7;
    border-radius: 8px;
    padding: 12px 16px;
    margin: 8px 0;
    font-size: 0.9rem;
}

/* Button Styling */
.stButton > button {
    background: linear-gradient(135deg, #a855f7 0%, #6366f1 100%) !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 12px !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    padding: 12px 24px !important;
    width: 100% !important;
    box-shadow: 0 4px 20px rgba(168, 85, 247, 0.3) !important;
    transition: all 0.3s ease !important;
}
.stButton > button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 25px rgba(168, 85, 247, 0.5) !important;
}

/* Inputs styling */
.stTextInput input {
    background: #161523 !important;
    border: 1px solid #2e2b44 !important;
    border-radius: 10px !important;
    color: #f1f5f9 !important;
}
.stTextInput input:focus {
    border-color: #a855f7 !important;
    box-shadow: 0 0 0 1px #a855f7 !important;
}

hr {
    border-color: #1e1c33 !important;
}
</style>
""", unsafe_allow_html=True)


# ── 2. LOAD DATASET & LIGHTWEIGHT NLP MODEL ──────────────────────────────────
@st.cache_data
def load_dataset():
    return pd.read_csv('spotify_data_combined.csv')

@st.cache_resource
def load_nlp_pipeline():
    # Fast & memory-friendly Zero-Shot Classifier
    return pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")

df = load_dataset()
nlp_classifier = load_nlp_pipeline()


# ── 3. OPTIMIZED ZERO-SHOT EMOTION MAPPINGS ───────────────────────────────────
CANDIDATE_LABELS = [
    "annoyed and stressed about work or duties",
    "sad, tired, and melancholic",
    "happy, cheerful, and excited",
    "calm, peaceful, and relaxed",
    "romantic and warm",
    "energetic and ready to party"
]

ZERO_SHOT_MAP = {
    "annoyed and stressed about work or duties": {'valence': 0.15, 'energy': 0.85, 'label': 'anger'},
    "sad, tired, and melancholic":               {'valence': 0.15, 'energy': 0.25, 'label': 'sadness'},
    "happy, cheerful, and excited":             {'valence': 0.85, 'energy': 0.80, 'label': 'joy'},
    "calm, peaceful, and relaxed":              {'valence': 0.50, 'energy': 0.30, 'label': 'neutral'},
    "romantic and warm":                        {'valence': 0.80, 'energy': 0.45, 'label': 'love'},
    "energetic and ready to party":             {'valence': 0.75, 'energy': 0.90, 'label': 'surprise'},
}

EMOTION_VECTORS = {
    'happy':    {'valence': 0.85, 'energy': 0.80},
    'sad':      {'valence': 0.15, 'energy': 0.25},
    'angry':    {'valence': 0.10, 'energy': 0.90},
    'disgust':  {'valence': 0.20, 'energy': 0.85},
    'neutral':  {'valence': 0.50, 'energy': 0.40},
    'surprise': {'valence': 0.70, 'energy': 0.85},
    'fear':     {'valence': 0.20, 'energy': 0.75},
}


# ── 4. HEADER ─────────────────────────────────────────────────────────────────
st.markdown('<div class="hero-title">🎵 SongMind AI</div>', unsafe_allow_html=True)
st.markdown('<div class="hero-sub">Multimodal Recommendation Engine • Computer Vision + Zero-Shot NLP</div>', unsafe_allow_html=True)


# ── 5. INPUT SECTION ──────────────────────────────────────────────────────────
st.markdown("### 📸 Step 1: Capture Facial Expression")
picture = st.camera_input("Take a quick webcam selfie")

st.markdown("### 💬 Step 2: Chat Your Mood")
user_text = st.text_input(
    "How is your day going?",
    placeholder="e.g., 'Got work to do on the weekend by my boss...', 'Feeling relaxed with coffee'",
)

st.markdown("<br>", unsafe_allow_html=True)
analyze = st.button("🧠 Analyze Mood & Generate Playlist")


# ── 6. MULTIMODAL INFERENCE & RECOMMENDATION ─────────────────────────────────
if analyze:
    if not picture and not user_text:
        st.warning("⚠️ Please snap a picture or enter a chat message to begin.")
    else:
        v_scores, e_scores = [], []

        # --- A. COMPUTER VISION (DeepFace) ---
        if picture:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".jpg") as temp_file:
                temp_file.write(picture.getvalue())
                temp_path = temp_file.name

            try:
                cv_result = DeepFace.analyze(img_path=temp_path, actions=['emotion'], enforce_detection=False)
                face_emotion = cv_result[0]['dominant_emotion'].lower()

                face_vec = EMOTION_VECTORS.get(face_emotion, EMOTION_VECTORS['neutral'])
                v_scores.append(face_vec['valence'])
                e_scores.append(face_vec['energy'])

                st.markdown(
                    f'<div class="status-box">📸 <b>Facial Expression:</b> Detected <span style="color:#a855f7;">{face_emotion.capitalize()}</span> '
                    f'(Valence: <code>{face_vec["valence"]}</code>, Energy: <code>{face_vec["energy"]}</code>)</div>',
                    unsafe_allow_html=True
                )
            except Exception as ex:
                st.error(f"Error analyzing image: {ex}")
            finally:
                os.remove(temp_path)

        # --- B. NATURAL LANGUAGE PROCESSING (Zero-Shot) ---
        if user_text:
            try:
                result = nlp_classifier(user_text, candidate_labels=CANDIDATE_LABELS)
                top_label = result['labels'][0]

                vec = ZERO_SHOT_MAP[top_label]
                v_scores.append(vec['valence'])
                e_scores.append(vec['energy'])

                st.markdown(
                    f'<div class="status-box">💬 <b>Text Sentiment:</b> Interpreted <span style="color:#6366f1;">{vec["label"].capitalize()}</span> '
                    f'(Valence: <code>{vec["valence"]}</code>, Energy: <code>{vec["energy"]}</code>)</div>',
                    unsafe_allow_html=True
                )
            except Exception as ex:
                st.error(f"Error analyzing text sentiment: {ex}")

        # --- C. MULTIMODAL FUSION ---
        target_valence = float(np.mean(v_scores))
        target_energy = float(np.mean(e_scores))

        st.markdown("<br>", unsafe_allow_html=True)
        cols = st.columns(2)
        cols[0].metric("Target Valence (Happiness)", f"{target_valence:.2f}")
        cols[1].metric("Target Energy (Intensity)", f"{target_energy:.2f}")

        # --- D. COSINE SIMILARITY MATRIX MATCHING ---
        user_vector = np.array([[target_valence, target_energy]])
        song_vectors = df[['valence', 'energy']].values

        similarities = cosine_similarity(user_vector, song_vectors)[0]

        df_copy = df.copy()
        df_copy['match_score'] = (similarities * 100).round(1)
        top_recommendations = df_copy.sort_values(by='match_score', ascending=False).head(5)

        # --- E. SLEEK RECOMMENDATION CARDS ---
        st.markdown("---")
        st.markdown("### 🎧 Top 5 Personal Recommendations")

        for idx, row in top_recommendations.iterrows():
            st.markdown(f"""
            <div class="glass-card">
                <div style="display: flex; align-items: center; justify-content: space-between; gap: 16px;">
                    <img src="{row['Cover Image']}" style="width: 68px; height: 68px; border-radius: 10px; object-fit: cover;">
                    <div style="flex-grow: 1;">
                        <div style="font-family: 'Syne', sans-serif; font-size: 1.1rem; font-weight: 700; color: #f8fafc;">{row['Song Name']}</div>
                        <div style="font-size: 0.9rem; color: #94a3b8; margin-top: 2px;">{row['Artists']}</div>
                        <div style="font-size: 0.75rem; color: #a855f7; margin-top: 6px;">🏷️ {row['mood']} • Valence: {row['valence']} • Energy: {row['energy']}</div>
                    </div>
                    <div style="text-align: right;">
                        <div style="font-family: 'Syne', sans-serif; font-size: 1.4rem; font-weight: 800; color: #a855f7;">{row['match_score']}%</div>
                        <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Match</div>
                        <a href="{row['Spotify URL']}" target="_blank" style="display: inline-block; margin-top: 6px; font-size: 0.8rem; color: #6366f1; text-decoration: none; font-weight: 600;">▶ Listen</a>
                    </div>
                </div>
            </div>
            """, unsafe_allow_html=True)

Overwriting app.py


## 🌐 Step 6: Deploy Application via Localtunnel
Retrieve the Colab instance's public IP endpoint password and execute Streamlit over Localtunnel port 8501.

In [ ]:
# Fetch Endpoint IP Password for Localtunnel authentication
!wget -q -O - ipv4.icanhazip.com

34.6.97.96


In [ ]:
# Install Localtunnel tool globally
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠙

In [ ]:
# Launch Streamlit and open tunnel
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧your url is: https://rich-clouds-notice.loca.lt
2026-07-25 21:00:03.037 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.6.97.96:8501

Loading weights: 100% 104/104 [00:00<00:00, 468.64it/s]
  Stopping...
^C
